In [11]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.io
from alphacsc import learn_d_z # noqa
from alphacsc.viz.callback import plot_data
from pathlib import Path
import sys

In [35]:
dossier_actuel = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()
racine_projet = dossier_actuel.parent
chemin_fichier = racine_projet / "data" / "ECGData.mat"
sys.path.append(str(racine_projet))
mat=scipy.io.loadmat(chemin_fichier)#on importe les données
xdata=mat['ECGData']

signaux=xdata[0][0][0]#préparation des données
X=signaux[0]
ARR=96
CHF=ARR+30
T=1/128#période d'acquisition 
t=np.arange(0,len(X)*T,T)

In [ ]:
reg = 0.1 # paramètres de fonctionnement
n_iter = 50

n_atoms=3
n_times_atom=45#variables à ajuster pour un résultat optimal


_, _, d_nsr, z_nsr, _ = learn_d_z(
    signaux[:][CHF:], n_atoms, n_times_atom, reg=reg, n_iter=n_iter,
    solver_d_kwargs=dict(factr=100),
    n_jobs=55, verbose=1)#calcul des atomes et temps d'activations pour les signaux de types nsr

_, _, d_arr, z_arr, _ = learn_d_z(
    signaux[:][:ARR], n_atoms, n_times_atom, reg=reg, n_iter=n_iter,
    solver_d_kwargs=dict(factr=100),
    n_jobs=55, verbose=1)#calcul des atomes et temps d'activations pour les signaux de types arr

_, _, d_chf, z_chf, _ = learn_d_z(
    signaux[:][ARR:CHF], n_atoms, n_times_atom, reg=reg, n_iter=n_iter,
    solver_d_kwargs=dict(factr=100),
    n_jobs=55, verbose=1)#calcul des atomes et temps d'activations pour les signaux de types chf

In [ ]:
plt.title('Atomes NSR')
plt.plot(d_nsr.T)
plt.show()

plt.title('Atomes ARR')
plt.plot(d_arr.T)
plt.show()

plt.title('Atomes CHF')
plt.plot(d_chf.T)
plt.show()

plt.title("Temps d'activations NSR")
plot_data([z[:10] for z in z_nsr], ['stem'] * n_atoms)
plt.show()
plt.title("Temps d'activations CHF")
plot_data([z[:10] for z in z_chf], ['stem'] * n_atoms)
plt.show()
plt.title("Temps d'activations ARR")
plot_data([z[:10] for z in z_arr], ['stem'] * n_atoms)
plt.show()